<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 16


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс PaymentMethod в C#, который будет представлять
различные способы оплаты. На основе этого класса разработать 2-3 производных
класса, демонстрирующих принципы наследования и полиморфизма. В каждом из
классов должны быть реализованы новые атрибуты и методы, а также
переопределены некоторые методы базового класса для демонстрации
полиморфизма.
Требования к базовому классу PaymentMethod:
• Атрибуты: ID способа оплаты (PaymentMethodId), Название способа оплаты
(MethodName), Минимальная сумма (MinAmount).
• Методы:
o ProcessPayment(decimal amount): метод для обработки платежа
указанной суммы.
o CheckMinimumAmount(decimal amount): метод для проверки
минимальной суммы платежа.
o GetPaymentDetails(): метод для получения деталей способа оплаты.
Требования к производным классам:
1. ОнлайнОплата (OnlinePayment): Должен содержать дополнительные
атрибуты, такие как URL платежной системы (PaymentUrl).
Метод ProcessPayment() должен быть переопределен для включения URL
платежной системы в процесс оплаты.
2. БанковскийПеревод (BankTransfer): Должен содержать дополнительные
атрибуты, такие как Банковские данные (BankData).
Метод CheckMinimumAmount() должен быть переопределен для проверки
минимальной суммы платежа с учетом банковских комиссий.
3. Наличные (CashPayment) (если требуется третий класс): Должен содержать
дополнительные атрибуты, такие как Место выдачи наличных
(CashPickupPoint). Метод GetPaymentDetails() должен быть переопределен
для отображения места выдачи наличных.

#### Дополнительное задание
Добавьте к сущестующим классам конструктора классов с использованием гетторов и сетторов и реализуйте взаимодействие объектов между собой

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [2]:
using System;
using System.Collections.Generic;

// 1. Интерфейс с геттерами и сеттерами
public interface IPaymentMethod
{
    int PaymentMethodId { get; set; }
    string MethodName { get; set; }
    decimal MinAmount { get; set; }
    IPaymentMethod? FallbackMethod { get; set; } // Ссылка на другой способ оплаты

    bool CheckMinimumAmount(decimal amount);
    string ProcessPayment(decimal amount);
    string GetPaymentDetails();
}

// 2. Базовый абстрактный класс
public abstract class PaymentMethod : IPaymentMethod
{
    private decimal _minAmount;

    public int PaymentMethodId { get; set; }
    public string MethodName { get; set; }

    // Свойство с валидацией в сеттере
    public decimal MinAmount
    {
        get => _minAmount;
        set
        {
            if (value < 0)
                throw new ArgumentException("Минимальная сумма не может быть отрицательной.");
            _minAmount = value;
        }
    }

    // Резервный способ оплаты (взаимодействие объектов способов оплаты)
    public IPaymentMethod? FallbackMethod { get; set; }

    // Конструктор
    protected PaymentMethod(int id, string methodName, decimal minAmount)
    {
        PaymentMethodId = id;
        MethodName = methodName;
        MinAmount = minAmount; // Инициализация через сеттер с валидацией
    }

    public virtual bool CheckMinimumAmount(decimal amount) => amount >= MinAmount;

    public virtual string ProcessPayment(decimal amount)
    {
        if (!CheckMinimumAmount(amount))
        {
            // Взаимодействие с резервным объектом, если свой не подходит
            if (FallbackMethod != null)
            {
                return $"Сумма {amount:C} меньше минимума ({MinAmount:C}) для '{MethodName}'. " +
                       $"Переключение на резервный способ...\n   -> {FallbackMethod.ProcessPayment(amount)}";
            }
            return $"Ошибка: сумма {amount:C} меньше минимальной ({MinAmount:C}).";
        }
        return $"Оплата {amount:C} через '{MethodName}' выполнена.";
    }

    public virtual string GetPaymentDetails()
        => $"ID: {PaymentMethodId}, Метод: {MethodName}, Мин. сумма: {MinAmount:C}";
}

// 3. Онлайн-оплата
public class OnlinePayment : PaymentMethod
{
    private string _paymentUrl = string.Empty;

    public string PaymentUrl
    {
        get => _paymentUrl;
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("URL платежной системы не может быть пустым.");
            _paymentUrl = value;
        }
    }

    public OnlinePayment(int id, string methodName, decimal minAmount, string paymentUrl)
        : base(id, methodName, minAmount)
    {
        PaymentUrl = paymentUrl; // Вызов сеттера
    }

    public override string ProcessPayment(decimal amount)
    {
        if (!CheckMinimumAmount(amount))
        {
            if (FallbackMethod != null)
            {
                return $"Сумма {amount:C} меньше минимума ({MinAmount:C}) для '{MethodName}'. " +
                       $"Переключение на резервный способ...\n   -> {FallbackMethod.ProcessPayment(amount)}";
            }
            return $"Ошибка: сумма {amount:C} меньше минимальной ({MinAmount:C}).";
        }
        return $"Онлайн-оплата {amount:C} через '{MethodName}'. Перейдите: {PaymentUrl}";
    }

    public override string GetPaymentDetails()
        => base.GetPaymentDetails() + $", URL: {PaymentUrl}";
}

// 4. Банковский перевод
public class BankTransfer : PaymentMethod
{
    public string BankData { get; set; }
    public decimal CommissionPercent { get; set; }

    public BankTransfer(int id, string methodName, decimal minAmount,
                        string bankData, decimal commissionPercent = 1.5m)
        : base(id, methodName, minAmount)
    {
        BankData = bankData;
        CommissionPercent = commissionPercent;
    }

    public override bool CheckMinimumAmount(decimal amount)
    {
        decimal commission = amount * CommissionPercent / 100m;
        return (amount + commission) >= MinAmount;
    }

    public override string ProcessPayment(decimal amount)
    {
        if (!CheckMinimumAmount(amount))
        {
            if (FallbackMethod != null)
            {
                return $"Сумма {amount:C} с комиссией меньше минимума ({MinAmount:C}) для '{MethodName}'. " +
                       $"Переключение на резервный способ...\n   -> {FallbackMethod.ProcessPayment(amount)}";
            }
            return $"Ошибка: сумма {amount:C} с комиссией {CommissionPercent}% меньше минимальной ({MinAmount:C}).";
        }
        decimal commission = amount * CommissionPercent / 100m;
        return $"Банковский перевод {amount:C} (комиссия {commission:C}). Реквизиты: {BankData}";
    }

    public override string GetPaymentDetails()
        => base.GetPaymentDetails() + $", Банк. данные: {BankData}, Комиссия: {CommissionPercent}%";
}

// 5. Наличные
public class CashPayment : PaymentMethod
{
    public string CashPickupPoint { get; set; }

    public CashPayment(int id, string methodName, decimal minAmount, string cashPickupPoint)
        : base(id, methodName, minAmount)
    {
        CashPickupPoint = cashPickupPoint;
    }

    public override string GetPaymentDetails()
        => base.GetPaymentDetails() + $", Место выдачи наличных: {CashPickupPoint}";
}

// 6. Класс Заказа (демонстрация взаимодействия объекта Order и объекта IPaymentMethod)
public class Order
{
    public int OrderId { get; set; }
    public decimal TotalAmount { get; set; }
    public IPaymentMethod SelectedPaymentMethod { get; set; }

    public Order(int orderId, decimal totalAmount, IPaymentMethod paymentMethod)
    {
        OrderId = orderId;
        TotalAmount = totalAmount;
        SelectedPaymentMethod = paymentMethod;
    }

    public void PayOrder()
    {
        Console.WriteLine($"[Заказ #{OrderId}] Попытка оплаты на сумму {TotalAmount:C}");
        Console.WriteLine($"Выбранный метод: {SelectedPaymentMethod.MethodName}");
        
        // Взаимодействие объекта Order с объектом Способа оплаты
        string status = SelectedPaymentMethod.ProcessPayment(TotalAmount);
        Console.WriteLine($"Результат: {status}\n");
    }
}

// 7. Точка входа (Демонстрация)
public static class PaymentDemo
{
    public static void Run()
    {
        // Создание объектов способов оплаты
        var cash = new CashPayment(3, "Наличные", 10m, "г. Москва, ул. Ленина, 10");
        var online = new OnlinePayment(1, "Онлайн-оплата картой", 500m, "https://pay.example.com/checkout");
        var bank = new BankTransfer(2, "Банковский перевод", 1000m, "IBAN: DE12 3456 7890", 2.0m);

        // Настройка взаимодействия между объектами (резервная цепочка)
        online.FallbackMethod = cash; // Если онлайн не пройдет по сумме, переключиться на наличные
        bank.FallbackMethod = cash;

        Console.WriteLine("=== Информация о способах оплаты ===\n");
        Console.WriteLine(online.GetPaymentDetails());
        Console.WriteLine(bank.GetPaymentDetails());
        Console.WriteLine(cash.GetPaymentDetails());

        Console.WriteLine("\n=== Взаимодействие через объекты Заказов (Order) ===\n");

        // Заказ #1: Проходит по минимальной сумме для онлайн-оплаты (500)
        Order order1 = new Order(101, 750m, online);
        order1.PayOrder();

        // Заказ #2: Сумма (100) ниже минимума для онлайн-оплаты (500).
        // Произойдет авто-переключение на резервный способ (CashPayment).
        Order order2 = new Order(102, 100m, online);
        order2.PayOrder();

        // Изменение свойства через сеттер с проверкой работы
        online.MinAmount = 200m;
        Console.WriteLine($"--> Минимальная сумма для '{online.MethodName}' изменена на {online.MinAmount:C}\n");

        // Заказ #3: Повторная попытка оплаты на 300
        Order order3 = new Order(103, 300m, online);
        order3.PayOrder();
    }
}

PaymentDemo.Run();

=== Информация о способах оплаты ===

ID: 1, Метод: Онлайн-оплата картой, Мин. сумма: ¤500.00, URL: https://pay.example.com/checkout
ID: 2, Метод: Банковский перевод, Мин. сумма: ¤1,000.00, Банк. данные: IBAN: DE12 3456 7890, Комиссия: 2.0%
ID: 3, Метод: Наличные, Мин. сумма: ¤10.00, Место выдачи наличных: г. Москва, ул. Ленина, 10

=== Взаимодействие через объекты Заказов (Order) ===

[Заказ #101] Попытка оплаты на сумму ¤750.00
Выбранный метод: Онлайн-оплата картой
Результат: Онлайн-оплата ¤750.00 через 'Онлайн-оплата картой'. Перейдите: https://pay.example.com/checkout

[Заказ #102] Попытка оплаты на сумму ¤100.00
Выбранный метод: Онлайн-оплата картой
Результат: Сумма ¤100.00 меньше минимума (¤500.00) для 'Онлайн-оплата картой'. Переключение на резервный способ...
   -> Оплата ¤100.00 через 'Наличные' выполнена.

--> Минимальная сумма для 'Онлайн-оплата картой' изменена на ¤200.00

[Заказ #103] Попытка оплаты на сумму ¤300.00
Выбранный метод: Онлайн-оплата картой
Результат: Онлайн-о